# Train Cheating Risk Model
This notebook trains a Temporal LSTM BiLSTM sequence classification model to detect cheating patterns over 30-frame sequence windows.
Features: [yaw, pitch, roll, gaze_x, gaze_y, face_count_norm]

In [ ]:
import os
import sys
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from pathlib import Path

# Add root project to path to access local files if needed
ROOT = Path("../../").resolve()
sys.path.insert(0, str(ROOT))

## 1. Load Dataset
You mentioned you have the datasets link - if you have pre-processed features ready, load them here. Otherwise, we can run the local `preprocess_cheating.py` synthetics.

In [ ]:
# NOTE: Insert your dataset link/path below. 
DATASET_LINK = "YOUR_DATASET_URL_HERE"
DATA_DIR = ROOT / "data" / "cheating_processed"

def ensure_data():
    if not DATA_DIR.exists():
        print("Data not found! Generating synthetic dataset via preprocess_cheating.py...")
        import importlib
        sys.path.append(str(ROOT / 'data'))
        from data.preprocess_cheating import generate_cheating_dataset
        generate_cheating_dataset(str(DATA_DIR), honest_count=4000, cheat_per_type=1000)
    else:
        print(f"Data found at {DATA_DIR}")

ensure_data()

def load_split(split): 
    X = np.load(DATA_DIR / split / "sequences.npy")
    y = np.load(DATA_DIR / split / "labels.npy")
    return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

X_train, y_train = load_split("train")
X_val, y_val = load_split("val")
X_test, y_test = load_split("test")

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=128, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=128, shuffle=False)

## 2. Define the Model

In [ ]:
class CheatingLSTM(nn.Module):
    def __init__(self, input_dim=6, hidden_dim=64, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers,
                            batch_first=True, bidirectional=True, dropout=0.3)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1)
        )
        
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last_hidden = lstm_out[:, -1, :] 
        logits = self.classifier(last_hidden)
        return logits
        
    def predict_probability(self, x):
        with torch.no_grad():
            logits = self(x)
            return torch.sigmoid(logits)

device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
model = CheatingLSTM().to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.BCEWithLogitsLoss()

## 3. Training Loop

In [ ]:
epochs = 25
best_loss = float('inf')

out_dir = Path("weights")
out_dir.mkdir(parents=True, exist_ok=True)
save_path = out_dir / "cheating_lstm.pth"

print(f"Training on {device}...")
for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    for X, y in train_loader:
        X, y = X.to(device), y.unsqueeze(1).to(device)
        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        
    # Validation
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for X, y in val_loader:
            X, y = X.to(device), y.unsqueeze(1).to(device)
            logits = model(X)
            loss = criterion(logits, y)
            val_loss += loss.item()
            
            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).float()
            correct += (preds == y).sum().item()
            total += y.size(0)
            
    val_acc = correct / total
    print(f"Epoch {epoch+1:02d}/{epochs} | Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {val_loss/len(val_loader):.4f} | Val Acc: {val_acc:.4f}")
    
    if val_loss < best_loss:
        best_loss = val_loss
        torch.save(model.state_dict(), save_path)

print(f"Training complete. Best model saved to {save_path}")